# Phase 2 - Preprocessing Pipeline

This notebook runs or validates the chunked preprocessing pipeline. The production logic stays in `src/preprocessing.py`; this notebook is only an interactive wrapper.

In [1]:
# Cell 1 - Set up imports and make the project package importable from notebooks.
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


Project root: E:\Paper Multiclass-Intrusion-Detection-System
Config path: E:\Paper Multiclass-Intrusion-Detection-System\configs\config.yaml


In [2]:
# Cell 2 - Load the shared YAML config and resolve the main artifact paths.
from src.data_loading import load_config, resolve_project_path

config = load_config(CONFIG_PATH)
processed_dir = resolve_project_path(config["paths"]["processed_dir"], PROJECT_ROOT)
models_dir = resolve_project_path(config["paths"]["models_dir"], PROJECT_ROOT)
metrics_dir = resolve_project_path(config["paths"]["metrics_dir"], PROJECT_ROOT)
figures_dir = resolve_project_path(config["paths"]["figures_dir"], PROJECT_ROOT)

print(f"Expected feature count: {config['data']['expected_feature_count']}")
print(f"Expected classes: {config['data']['expected_num_classes']}")


Expected feature count: 69
Expected classes: 10


In [3]:
# Cell 3 - Check whether the raw dataset and expected processed outputs are available.
raw_dataset = resolve_project_path(config["paths"]["raw_dataset"], PROJECT_ROOT)
required_processed_outputs = [
    processed_dir / "X_train.npy",
    processed_dir / "X_test.npy",
    processed_dir / "y_train.npy",
    processed_dir / "y_test.npy",
    processed_dir / "minmax_scaler.joblib",
    processed_dir / "label_mapping.json",
]

print(f"Raw dataset exists: {raw_dataset.exists()} -> {raw_dataset}")
for path in required_processed_outputs:
    print(f"{path.name}: {path.exists()}")


Raw dataset exists: True -> E:\Paper Multiclass-Intrusion-Detection-System\data\raw\CIC-ToN-IoT.csv
X_train.npy: True
X_test.npy: True
y_train.npy: True
y_test.npy: True
minmax_scaler.joblib: True
label_mapping.json: True


In [4]:
# Cell 4 - Run preprocessing only if needed, or force a rebuild for a full rerun.
from src.preprocessing import preprocess_dataset

FORCE_REBUILD = False
report_path = metrics_dir / "preprocessing_report.json"
needs_preprocessing = FORCE_REBUILD or not report_path.exists() or not all(path.exists() for path in required_processed_outputs)

if needs_preprocessing:
    preprocessing_report = preprocess_dataset(CONFIG_PATH)
else:
    with open(report_path, "r", encoding="utf-8") as file:
        preprocessing_report = json.load(file)

print(json.dumps(preprocessing_report, indent=2))


{
  "total_rows": 5351760,
  "valid_rows": 5350583,
  "invalid_rows_dropped": 1177,
  "train_rows": 4280466,
  "test_rows": 1070117,
  "feature_count": 69,
  "test_size": 0.2,
  "random_state": 42,
  "scaler_fit_scope": "train_only",
  "max_split_percentage_delta": 4.672838196390777e-05,
  "split_distribution_path": "E:\\Paper Multiclass-Intrusion-Detection-System\\results\\metrics\\split_class_distribution.csv",
  "X_train_shape": [
    4280466,
    69
  ],
  "X_test_shape": [
    1070117,
    69
  ],
  "y_train_shape": [
    4280466
  ],
  "y_test_shape": [
    1070117
  ]
}


In [5]:
# Cell 5 - Load processed arrays as memory maps and verify their shapes without loading all data into RAM.
x_train = np.load(processed_dir / "X_train.npy", mmap_mode="r")
x_test = np.load(processed_dir / "X_test.npy", mmap_mode="r")
y_train = np.load(processed_dir / "y_train.npy", mmap_mode="r")
y_test = np.load(processed_dir / "y_test.npy", mmap_mode="r")

shape_summary = pd.DataFrame([
    {"array": "X_train", "shape": tuple(x_train.shape), "dtype": str(x_train.dtype)},
    {"array": "X_test", "shape": tuple(x_test.shape), "dtype": str(x_test.dtype)},
    {"array": "y_train", "shape": tuple(y_train.shape), "dtype": str(y_train.dtype)},
    {"array": "y_test", "shape": tuple(y_test.shape), "dtype": str(y_test.dtype)},
])
shape_summary


,array,shape,dtype
0,X_train,"(4280466, 69)",float32
1,X_test,"(1070117, 69)",float32
2,y_train,"(4280466,)",int16
3,y_test,"(1070117,)",int16


In [6]:
# Cell 6 - Inspect stratified train/test class percentages saved by the preprocessing pipeline.
split_distribution = pd.read_csv(metrics_dir / "split_class_distribution.csv")
split_distribution.pivot(index="class", columns="split", values="percentage").round(6)


split,test,train
class,,
Backdoor,0.507328,0.507328
Benign,46.986638,46.986637
DDoS,0.003738,0.003785
DoS,0.002710,0.002710
Injection,5.189993,5.190019
MITM,0.009625,0.009672
Password,6.358370,6.358326
Ransomware,0.095317,0.095270
Scanning,0.676655,0.676655
